# NAFNet evaluation

In [1]:
%pip install torch numpy pillow scikit-image tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
"""Evaluate official NAFNet checkpoints on paired RGB PNGs (range 0..1).

Install: python -m pip install torch numpy pillow scikit-image tqdm
Run: python nafnet_eval.py --data-dir C:/path/to/DIV2K_Restoration
"""
from pathlib import Path
import argparse
import csv
import importlib
import sys
import urllib.request

COMMIT = "2b4af71ebe098a92a75910c233a3965a3e93ede4"


def official_architecture(cache_dir):
    # Keep upstream layers, removing only BasicSR package-wide dependencies.
    package = cache_dir / "nafnet_official"
    package.mkdir(parents=True, exist_ok=True)
    (package / "__init__.py").touch()
    for filename in ["arch_util.py", "local_arch.py", "NAFNet_arch.py"]:
        destination = package / filename
        if not destination.is_file():
            url = (
                f"https://raw.githubusercontent.com/megvii-research/NAFNet/{COMMIT}"
                f"/basicsr/models/archs/{filename}"
            )
            with urllib.request.urlopen(url, timeout=60) as response:
                source = response.read().decode("utf-8")
            source = source.replace("from basicsr.utils import get_root_logger", "# Unused BasicSR logger import removed for standalone evaluation.")
            source = source.replace("from basicsr.models.archs.arch_util import LayerNorm2d", "from .arch_util import LayerNorm2d")
            source = source.replace("from basicsr.models.archs.local_arch import Local_Base", "from .local_arch import Local_Base")
            compile(source, str(destination), "exec")
            destination.write_text(source, encoding="utf-8")
    sys.path.insert(0, str(cache_dir.resolve()))
    return importlib.import_module("nafnet_official.NAFNet_arch")


def load_network(arch, name, weights, device):
    import torch
    if not weights.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {weights}")
    if name == "SIDD":
        net = arch.NAFNet(img_channel=3, width=32, enc_blk_nums=[2, 2, 4, 8],
                          middle_blk_num=12, dec_blk_nums=[2, 2, 2, 2])
    else:
        # NAFNetLocal follows the official GoPro test configuration (TLC).
        net = arch.NAFNetLocal(img_channel=3, width=32,
                               enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1,
                               dec_blk_nums=[1, 1, 1, 1],
                               train_size=(1, 3, 256, 256), fast_imp=False)
    checkpoint = torch.load(weights, map_location="cpu", weights_only=True)
    if "params_ema" in checkpoint:
        state = checkpoint["params_ema"]
    elif "params" in checkpoint:
        state = checkpoint["params"]
    else:
        state = checkpoint
    state = {key.removeprefix("module."): value for key, value in state.items()}
    net.load_state_dict(state, strict=True)
    return net.to(device).eval()


def evaluate(data_dir, weights, output_dir, categories=("noise", "blur", "noise_blur"),
             split="test", device_name=None):
    import numpy as np
    import torch
    from PIL import Image
    from skimage.metrics import structural_similarity
    from tqdm.auto import tqdm

    torch.set_num_threads(min(torch.get_num_threads(), 4))
    device = torch.device(device_name or ("cuda" if torch.cuda.is_available() else "cpu"))
    output_dir.mkdir(parents=True, exist_ok=True)
    # Validate pairs before downloading layers or loading large networks.
    pairs = {}
    for category in categories:
        files = sorted((data_dir / split / category).glob("*.png"))
        if not files:
            raise FileNotFoundError(f"No PNG images: {data_dir / split / category}")
        pairs[category] = [(p, data_dir / split / "clean" / p.name) for p in files]
        for _, target in pairs[category]:
            if not target.is_file():
                raise FileNotFoundError(f"Missing paired clean image: {target}")

    arch = official_architecture(output_dir / ("source_" + COMMIT))
    rows, summaries = [], []
    print("Device:", device, "| RGB float [0,1], no resizing or border crop", flush=True)
    for name, checkpoint in weights.items():
        net = load_network(arch, name, Path(checkpoint), device)
        print(f"Loaded {name}: all checkpoint keys matched", flush=True)
        with torch.inference_mode():
            for category, image_pairs in pairs.items():
                scores = []
                for source, target in tqdm(image_pairs, desc=f"{name}/{category}"):
                    with Image.open(source) as im:
                        degraded = np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0
                    with Image.open(target) as im:
                        clean = np.asarray(im.convert("RGB"), dtype=np.float32) / 255.0
                    if degraded.shape != clean.shape:
                        raise ValueError(f"Image shape mismatch: {source} / {target}")
                    if min(clean.shape[:2]) < 11:
                        raise ValueError(f"SSIM requires images at least 11x11: {target}")
                    x = torch.from_numpy(degraded.transpose(2, 0, 1).copy()).unsqueeze(0).to(device)
                    pred = net(x).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()
                    if not np.isfinite(pred).all():
                        raise ValueError(f"Non-finite prediction: {name}/{source.name}")
                    error = clean.astype(np.float64) - pred.astype(np.float64)
                    mae = float(np.mean(np.abs(error)))
                    mse = float(np.mean(error ** 2))
                    psnr = float(-10 * np.log10(mse)) if mse > 0 else float("inf")
                    ssim = float(structural_similarity(
                        clean, pred, data_range=1.0, channel_axis=-1,
                        gaussian_weights=True, sigma=1.5, use_sample_covariance=False,
                        win_size=11))
                    score = dict(MAE=mae, MSE=mse, PSNR=psnr, SSIM=ssim)
                    scores.append(score)
                    rows.append(dict(model=name, category=category, filename=source.name, **score))
                averages = {k: float(np.mean([s[k] for s in scores])) for k in scores[0]}
                summaries.append(dict(model=name, category=category, images=len(scores), **averages))
                print(f"{name:5s} | {category:12s} | MAE: {averages['MAE']:.6f} | "
                      f"MSE: {averages['MSE']:.6f} | PSNR: {averages['PSNR']:.2f} dB | "
                      f"SSIM: {averages['SSIM']:.4f}", flush=True)
        del net
        if device.type == "cuda":
            torch.cuda.empty_cache()
    for filename, records in [("per_image_metrics.csv", rows), ("summary_metrics.csv", summaries)]:
        with (output_dir / filename).open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=list(records[0]))
            writer.writeheader()
            writer.writerows(records)
    print("Results saved:", output_dir.resolve())
    return summaries




In [3]:
from pathlib import Path

# Windows paths. On Kaggle, upload both weights as Input and replace these paths.
DATA_DIR = Path(r"/kaggle/input/datasets/wisits/asdasdasda/DIV2K_Restoration")  # change to your dataset folder
SIDD_WEIGHTS = Path(r"/kaggle/input/models/wisits/aaaa/pytorch/default/1/NAFNet-SIDD-width32.pth")
GOPRO_WEIGHTS = Path(r"/kaggle/input/models/wisits/aaaa/pytorch/default/1/NAFNet-GoPro-width32.pth")
OUTPUT_DIR = Path("/kaggle/working/nafnet_eval_results")

results = evaluate(
    data_dir=DATA_DIR,
    weights={"SIDD": SIDD_WEIGHTS, "GoPro": GOPRO_WEIGHTS},
    output_dir=OUTPUT_DIR,
)


Device: cpu | RGB float [0,1], no resizing or border crop
Loaded SIDD: all checkpoint keys matched


SIDD/noise:   0%|          | 0/720 [00:00<?, ?it/s]

SIDD  | noise        | MAE: 0.030956 | MSE: 0.002212 | PSNR: 29.16 dB | SSIM: 0.8174


SIDD/blur:   0%|          | 0/720 [00:00<?, ?it/s]

SIDD  | blur         | MAE: 0.032279 | MSE: 0.003894 | PSNR: 29.54 dB | SSIM: 0.7406


...

GoPro | noise        | MAE: 0.491943 | MSE: 0.326881 | PSNR: 5.19 dB | SSIM: 0.0205


GoPro/blur:   0%|          | 0/720 [00:00<?, ?it/s]

GoPro | blur         | MAE: 0.053757 | MSE: 0.018391 | PSNR: 28.18 dB | SSIM: 0.7223


GoPro/noise_blur:   0%|          | 0/720 [00:00<?, ?it/s]

GoPro | noise_blur   | MAE: 0.496329 | MSE: 0.331198 | PSNR: 5.10 dB | SSIM: 0.0160
Results saved: /kaggle/working/nafnet_eval_results
